# 12 - Eventstream Demo

Generates simulated telemetry events for operating rooms (Operatiekamer) and sends them to a Fabric Eventstream.

**Purpose:** Demonstrate real-time data flow: Notebook → Eventstream → KQL Database → Ontology binding

## Prerequisites

1. Eventstream created with Custom App source
2. Connection string copied from Eventstream
3. KQL Database destination configured

## Configuration

In [ ]:
# ==============================================================================
# CONFIGURATION - Update these values
# ==============================================================================

# Eventstream Custom App connection string (from Eventstream portal)
# Format: Endpoint=sb://<namespace>.servicebus.windows.net/;SharedAccessKeyName=...;SharedAccessKey=...;EntityPath=...
EVENTSTREAM_CONNECTION_STRING = "<PASTE_YOUR_CONNECTION_STRING_HERE>"

# Operating rooms to simulate (should match your ontology instances)
OPERATIEKAMER_IDS = [
    "Operatiekamer_1",
    "Operatiekamer_2",
    "Operatiekamer_3",
]

# Simulation parameters
TEMPERATURE_UPDATE_INTERVAL_SECONDS = 60      # Temperature update every minute
STATUS_UPDATE_INTERVAL_SECONDS = 3600         # Status change every hour
DEMO_DURATION_MINUTES = 5                     # How long to run the demo
DEMO_FAST_MODE = True                         # If True, compress time for demo (updates every 5 sec)

## Install Dependencies

In [ ]:
# Install Azure Event Hubs SDK (Eventstream uses Event Hubs protocol)
%pip install azure-eventhub -q

## Event Generator

In [ ]:
import json
import random
import time
from datetime import datetime, timezone
from azure.eventhub import EventHubProducerClient, EventData

# Status values for operating rooms
STATUSES = [
    "Available",      # Room ready for next procedure
    "In Use",         # Surgery in progress
    "Cleaning",       # Post-procedure cleaning
    "Maintenance",    # Equipment maintenance/repair
]

# Status transition probabilities (realistic hospital flow)
STATUS_TRANSITIONS = {
    "Available": ["In Use", "Maintenance"],
    "In Use": ["Cleaning"],
    "Cleaning": ["Available"],
    "Maintenance": ["Available"],
}

# Room state tracking
room_states = {}

def initialize_rooms():
    """Initialize room states with random starting values."""
    for room_id in OPERATIEKAMER_IDS:
        room_states[room_id] = {
            "temperature": round(random.uniform(20.0, 22.0), 1),
            "status": random.choice(["Available", "In Use"]),
            "last_status_change": time.time(),
        }
    print(f"Initialized {len(room_states)} operating rooms")
    for room_id, state in room_states.items():
        print(f"  {room_id}: {state['status']}, {state['temperature']}°C")

def generate_temperature(room_id: str) -> float:
    """Generate realistic temperature with small variations."""
    current = room_states[room_id]["temperature"]
    # Small random walk, staying in 19-23°C range
    delta = random.uniform(-0.3, 0.3)
    new_temp = max(19.0, min(23.0, current + delta))
    room_states[room_id]["temperature"] = round(new_temp, 1)
    return room_states[room_id]["temperature"]

def maybe_update_status(room_id: str, status_interval: float) -> str:
    """Possibly transition to a new status based on time and transitions."""
    state = room_states[room_id]
    time_since_change = time.time() - state["last_status_change"]
    
    # Check if it's time for a possible status change
    if time_since_change >= status_interval:
        current_status = state["status"]
        possible_next = STATUS_TRANSITIONS.get(current_status, [current_status])
        
        # 70% chance to transition
        if random.random() < 0.7 and possible_next:
            new_status = random.choice(possible_next)
            state["status"] = new_status
            state["last_status_change"] = time.time()
            print(f"  ⚡ {room_id}: {current_status} → {new_status}")
    
    return state["status"]

def create_event(room_id: str, status_interval: float) -> dict:
    """Create a telemetry event for a room."""
    return {
        "operatiekamer_id": room_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "temperature_celsius": generate_temperature(room_id),
        "status": maybe_update_status(room_id, status_interval),
    }

print("Event generator functions loaded.")

## Send Events to Eventstream

In [ ]:
def send_events_batch(producer: EventHubProducerClient, events: list) -> None:
    """Send a batch of events to Eventstream."""
    event_data_batch = producer.create_batch()
    for event in events:
        event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)

def run_demo():
    """Run the event streaming demo."""
    if EVENTSTREAM_CONNECTION_STRING == "<PASTE_YOUR_CONNECTION_STRING_HERE>":
        print("❌ ERROR: Please configure EVENTSTREAM_CONNECTION_STRING first!")
        return
    
    # Demo mode timing
    if DEMO_FAST_MODE:
        temp_interval = 5      # Every 5 seconds
        status_interval = 30   # Every 30 seconds
        print("🚀 Running in FAST MODE (compressed time for demo)")
    else:
        temp_interval = TEMPERATURE_UPDATE_INTERVAL_SECONDS
        status_interval = STATUS_UPDATE_INTERVAL_SECONDS
    
    # Initialize
    initialize_rooms()
    
    # Create producer
    producer = EventHubProducerClient.from_connection_string(EVENTSTREAM_CONNECTION_STRING)
    
    print(f"\n📡 Starting event stream (duration: {DEMO_DURATION_MINUTES} minutes)")
    print(f"   Temperature updates: every {temp_interval}s")
    print(f"   Status changes: possible every {status_interval}s")
    print("   Press Ctrl+C to stop early\n")
    
    start_time = time.time()
    end_time = start_time + (DEMO_DURATION_MINUTES * 60)
    event_count = 0
    
    try:
        while time.time() < end_time:
            # Generate events for all rooms
            events = [create_event(room_id, status_interval) for room_id in OPERATIEKAMER_IDS]
            
            # Send batch
            send_events_batch(producer, events)
            event_count += len(events)
            
            # Log summary
            elapsed = time.time() - start_time
            print(f"[{elapsed:6.1f}s] Sent {len(events)} events (total: {event_count})")
            for e in events:
                print(f"         {e['operatiekamer_id']}: {e['temperature_celsius']}°C, {e['status']}")
            
            # Wait for next interval
            time.sleep(temp_interval)
            
    except KeyboardInterrupt:
        print("\n⏹️  Stopped by user")
    finally:
        producer.close()
        elapsed = time.time() - start_time
        print(f"\n✅ Demo complete: {event_count} events sent in {elapsed:.1f}s")

print("Run run_demo() to start streaming events.")

## Run Demo

In [ ]:
# Uncomment and run to start the demo
# run_demo()

## Quick Test (Single Batch)

In [ ]:
def send_single_batch():
    """Send a single batch of events for quick testing."""
    if EVENTSTREAM_CONNECTION_STRING == "<PASTE_YOUR_CONNECTION_STRING_HERE>":
        print("❌ ERROR: Please configure EVENTSTREAM_CONNECTION_STRING first!")
        return
    
    initialize_rooms()
    
    producer = EventHubProducerClient.from_connection_string(EVENTSTREAM_CONNECTION_STRING)
    events = [create_event(room_id, 9999) for room_id in OPERATIEKAMER_IDS]
    
    send_events_batch(producer, events)
    producer.close()
    
    print(f"\n✅ Sent {len(events)} events:")
    for e in events:
        print(f"   {json.dumps(e)}")

# Uncomment to test
# send_single_batch()

## Preview Events (No Send)

In [ ]:
# Preview what events would look like without sending
initialize_rooms()
print("\nSample events:")
for room_id in OPERATIEKAMER_IDS:
    event = create_event(room_id, 9999)
    print(json.dumps(event, indent=2))